<a href="https://colab.research.google.com/github/ardizzone88/vehicle_sales_analysis/blob/master/vehicle_sales_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚗 Vehicle Market Intelligence Dashboard
### End-to-End Data Analysis Project | NHTSA API + Synthetic Sales Data

---

**Author:** David Ardizzone
**Date:** 2026  
**Tools:** Python · Pandas · Plotly · Requests · NHTSA API (Free, no key needed)

---

## 📌 Project Overview

Este proyecto analiza el mercado automotriz utilizando:
- **NHTSA vPIC API** — catálogo oficial de vehículos del gobierno de EE.UU. (gratis, sin clave)
- **Datos de ventas sintéticos** — generados con distribuciones realistas basadas en tendencias reales del mercado

### 🎯 Business Questions
1. ¿Qué fabricantes dominan el mercado en volumen de ventas?
2. ¿Cómo evolucionaron las ventas de vehículos eléctricos vs. combustión?
3. ¿Cuáles son los segmentos más rentables?
4. ¿Qué patrones estacionales afectan las ventas?
5. ¿Cómo se correlacionan precio y volumen de ventas?

---

## ⚙️ 1. Setup & Instalación

In [1]:
# ── Instalar dependencias ──────────────────────────────────────────────────────
!pip install plotly pandas requests numpy kaleido --quiet

print("✅ Dependencias instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 725.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 1.9 MB/s eta 0:00:00
✅ Dependencias instaladas correctamente


In [2]:
# ── Imports ────────────────────────────────────────────────────────────────────
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import time
from datetime import datetime, timedelta
import json

warnings.filterwarnings('ignore')

# ── Configuración de estilos globales ─────────────────────────────────────────
TEMPLATE    = 'plotly_dark'
PRIMARY     = '#00D4FF'
SECONDARY   = '#FF6B6B'
ACCENT      = '#FFE66D'
FONT_FAMILY = 'Inter, Arial, sans-serif'

LAYOUT_DEFAULTS = dict(
    template    = TEMPLATE,
    font        = dict(family=FONT_FAMILY, size=13),
    margin      = dict(t=80, b=60, l=60, r=40),
    plot_bgcolor = 'rgba(0,0,0,0)',
)

print("✅ Imports y configuración listos")
print(f"📅 Análisis iniciado: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

✅ Imports y configuración listos
📅 Análisis iniciado: 2026-06-01 20:49


## 🌐 2. Extracción de Datos — NHTSA vPIC API

> **NHTSA vPIC** (Vehicle Product Information Catalog) es la API oficial del gobierno de EE.UU.  
> Endpoint base: `https://vpic.nhtsa.dot.gov/api/`  
> ✅ Completamente gratuita · Sin clave de API · Sin límite de requests

In [3]:
# ── Clase wrapper para la API ──────────────────────────────────────────────────
class NHTSAClient:
    """Cliente para la API NHTSA vPIC — gratuita, sin autenticación."""

    BASE_URL = 'https://vpic.nhtsa.dot.gov/api/vehicles'

    def _get(self, endpoint: str) -> list:
        """Realiza una petición GET y devuelve los resultados."""
        url = f'{self.BASE_URL}/{endpoint}&format=json'
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            return r.json().get('Results', [])
        except Exception as e:
            print(f'⚠️  Error en {url}: {e}')
            return []

    def get_all_makes(self) -> pd.DataFrame:
        """Devuelve todos los fabricantes registrados."""
        data = self._get('getallmakes?')
        return pd.DataFrame(data)[['Make_ID', 'Make_Name']].rename(
            columns={'Make_ID': 'make_id', 'Make_Name': 'make_name'})

    def get_models_for_make(self, make: str) -> pd.DataFrame:
        """Devuelve modelos disponibles para un fabricante."""
        data = self._get(f'getmodelsformake/{make}?')
        return pd.DataFrame(data) if data else pd.DataFrame()

    def get_vehicle_types_for_make(self, make: str) -> list:
        """Devuelve tipos de vehículos que produce un fabricante."""
        data = self._get(f'getvehicletypesformake/{make}?')
        return [d.get('VehicleTypeName', '') for d in data]

    def get_makes_for_vehicle_type(self, vtype: str) -> pd.DataFrame:
        """Devuelve fabricantes por tipo de vehículo."""
        data = self._get(f'getmakesforvehicletype/{vtype}?')
        return pd.DataFrame(data) if data else pd.DataFrame()


# ── Instanciar y obtener datos ─────────────────────────────────────────────────
client = NHTSAClient()

print('⏳ Consultando API NHTSA ...')
all_makes_df = client.get_all_makes()
print(f'✅ Fabricantes registrados en NHTSA: {len(all_makes_df):,}')
all_makes_df.head()

⏳ Consultando API NHTSA ...
✅ Fabricantes registrados en NHTSA: 12,261


,make_id,make_name
0,12858,#1 ALPINE CUSTOMS
1,4877,"1/OFF KUSTOMS, LLC"
2,11257,"102 IRONWORKS, INC."
3,12255,12832429 CANADA INC.
4,13053,137 INDUSTRIES INC.


In [4]:
# ── Obtener modelos de los top fabricantes ─────────────────────────────────────
TOP_MAKES = ['Toyota', 'Ford', 'Chevrolet', 'Honda', 'Tesla',
             'BMW', 'Mercedes-Benz', 'Audi', 'Hyundai', 'Kia',
             'Volkswagen', 'Nissan', 'Subaru', 'Jeep', 'Ram']

models_data = []
for make in TOP_MAKES:
    df = client.get_models_for_make(make)
    if not df.empty and 'Model_Name' in df.columns:
        for model in df['Model_Name'].tolist():
            models_data.append({'make': make, 'model': model})
    time.sleep(0.2)   # respetar rate limiting

models_df = pd.DataFrame(models_data)
print(f'✅ Modelos cargados: {len(models_df):,}')
print(f'🏭 Distribución por fabricante:')
print(models_df['make'].value_counts().to_string())

✅ Modelos cargados: 1,502
🏭 Distribución por fabricante:
make
Honda            459
BMW              258
Ford             168
Chevrolet        124
Ram              111
Mercedes-Benz     60
Toyota            58
Audi              55
Nissan            44
Volkswagen        40
Hyundai           40
Kia               27
Subaru            26
Jeep              24
Tesla              8


In [5]:
# ── Tipos de vehículos disponibles en la API ───────────────────────────────────
ev_makes_df  = client.get_makes_for_vehicle_type('Electric')
mpv_makes_df = client.get_makes_for_vehicle_type('Multipurpose Passenger Vehicle (MPV)')

print(f"⚡ Fabricantes con vehículos eléctricos registrados: {len(ev_makes_df)}")
if not ev_makes_df.empty:
    print(ev_makes_df[['MakeName']].head(10).to_string(index=False))

⚡ Fabricantes con vehículos eléctricos registrados: 0


## 🧪 3. Generación de Datos de Ventas Sintéticos

> **Nota metodológica:** Los datos de ventas son sintéticos, generados con distribuciones estadísticas  
> basadas en tendencias reales del mercado automotriz 2020-2024 (fuentes: Cox Automotive, Statista, BEA).  
> Esta técnica es estándar en proyectos de portfolio cuando los datos reales están detrás de paywalls.

In [6]:
np.random.seed(42)  # reproducibilidad

# ── Parámetros de mercado basados en datos reales ──────────────────────────────
MAKE_CONFIG = {
    'Toyota':         {'base_sales': 2200000, 'avg_price': 35000, 'segment': 'Mass Market',  'ev_share': 0.03},
    'Ford':           {'base_sales': 1900000, 'avg_price': 42000, 'segment': 'Mass Market',  'ev_share': 0.05},
    'Chevrolet':      {'base_sales': 1700000, 'avg_price': 38000, 'segment': 'Mass Market',  'ev_share': 0.04},
    'Honda':          {'base_sales': 1300000, 'avg_price': 31000, 'segment': 'Mass Market',  'ev_share': 0.01},
    'Tesla':          {'base_sales':  600000, 'avg_price': 58000, 'segment': 'EV/Tech',      'ev_share': 1.00},
    'BMW':            {'base_sales':  350000, 'avg_price': 68000, 'segment': 'Luxury',       'ev_share': 0.12},
    'Mercedes-Benz':  {'base_sales':  330000, 'avg_price': 75000, 'segment': 'Luxury',       'ev_share': 0.10},
    'Audi':           {'base_sales':  220000, 'avg_price': 62000, 'segment': 'Luxury',       'ev_share': 0.08},
    'Hyundai':        {'base_sales':  740000, 'avg_price': 29000, 'segment': 'Mass Market',  'ev_share': 0.06},
    'Kia':            {'base_sales':  680000, 'avg_price': 27000, 'segment': 'Mass Market',  'ev_share': 0.05},
    'Volkswagen':     {'base_sales':  370000, 'avg_price': 34000, 'segment': 'Mass Market',  'ev_share': 0.07},
    'Nissan':         {'base_sales':  850000, 'avg_price': 28000, 'segment': 'Mass Market',  'ev_share': 0.03},
    'Subaru':         {'base_sales':  680000, 'avg_price': 33000, 'segment': 'Mass Market',  'ev_share': 0.00},
    'Jeep':           {'base_sales':  740000, 'avg_price': 48000, 'segment': 'SUV/Truck',    'ev_share': 0.02},
    'Ram':            {'base_sales':  560000, 'avg_price': 52000, 'segment': 'SUV/Truck',    'ev_share': 0.01},
}

# ── Tendencia EV global año a año ─────────────────────────────────────────────
EV_GROWTH   = {2020: 1.0, 2021: 1.65, 2022: 2.55, 2023: 3.40, 2024: 4.10}
MARKET_TREND = {2020: 0.88, 2021: 0.94, 2022: 1.00, 2023: 1.05, 2024: 1.08}  # COVID dip

# ── Generar dataset de transacciones ──────────────────────────────────────────
records = []
for year in range(2020, 2025):
    for make, cfg in MAKE_CONFIG.items():

        # Modelos disponibles
        make_models = models_df[models_df['make'] == make]['model'].tolist()
        if not make_models:
            make_models = [f'{make} Model A', f'{make} Model B']

        annual_sales = int(cfg['base_sales'] * MARKET_TREND[year]
                           * np.random.uniform(0.92, 1.08))

        for month in range(1, 13):
            # Estacionalidad: Q4 y verano son fuertes
            seasonal = 1.0
            if month in [3, 4, 5]:  seasonal = 1.05
            if month in [7, 8]:     seasonal = 1.08
            if month in [11, 12]:   seasonal = 1.12
            if month in [1, 2]:     seasonal = 0.82

            monthly = int((annual_sales / 12) * seasonal * np.random.uniform(0.95, 1.05))

            # Distribución entre modelos del fabricante
            n_models  = min(len(make_models), 8)
            top_models = make_models[:n_models]
            weights    = np.random.dirichlet(np.ones(n_models) * 1.5)

            for model, w in zip(top_models, weights):
                model_sales = max(1, int(monthly * w))

                # Tipo de propulsión
                is_ev = np.random.random() < (cfg['ev_share'] * EV_GROWTH.get(year, 1))
                drive_type = 'Electric' if is_ev else np.random.choice(
                    ['Gasoline', 'Hybrid', 'Diesel'],
                    p=[0.78, 0.18, 0.04]
                )

                # Precio con varianza
                price_mult = 1.15 if is_ev else 1.0
                avg_price  = cfg['avg_price'] * price_mult * np.random.uniform(0.88, 1.15)

                records.append({
                    'year':       year,
                    'month':      month,
                    'date':       pd.Timestamp(year=year, month=month, day=1),
                    'make':       make,
                    'model':      model,
                    'segment':    cfg['segment'],
                    'drive_type': drive_type,
                    'units_sold': model_sales,
                    'avg_price':  round(avg_price, 2),
                    'revenue':    round(model_sales * avg_price, 2),
                })

df = pd.DataFrame(records)
df['quarter'] = df['date'].dt.quarter.map({1:'Q1', 2:'Q2', 3:'Q3', 4:'Q4'})
df['year_q']  = df['year'].astype(str) + '-' + df['quarter']

print(f'✅ Dataset generado: {len(df):,} registros')
print(f'📆 Período: {df["year"].min()} — {df["year"].max()}')
print(f'💰 Revenue total simulado: ${df["revenue"].sum()/1e9:.1f}B USD')
df.head()

✅ Dataset generado: 7,200 registros
📆 Período: 2020 — 2024
💰 Revenue total simulado: $2746.6B USD


,year,month,date,make,model,segment,drive_type,units_sold,avg_price,revenue,quarter,year_q
0,2020,1,2020-01-01,Toyota,Scion xA,Mass Market,Gasoline,17192,33560.77,5.769767e+08,Q1,2020-Q1
1,2020,1,2020-01-01,Toyota,Scion tC,Mass Market,Gasoline,31615,38219.91,1.208323e+09,Q1,2020-Q1
2,2020,1,2020-01-01,Toyota,Corolla,Mass Market,Gasoline,32747,36398.32,1.191936e+09,Q1,2020-Q1
3,2020,1,2020-01-01,Toyota,Prius,Mass Market,Gasoline,18830,32411.45,6.103077e+08,Q1,2020-Q1
4,2020,1,2020-01-01,Toyota,Scion xB,Mass Market,Hybrid,6223,39925.22,2.484547e+08,Q1,2020-Q1


## 🔍 4. Exploración de Datos (EDA)

In [7]:
# ── Resumen estadístico ────────────────────────────────────────────────────────
print('=' * 55)
print('         📊 RESUMEN DEL DATASET')
print('=' * 55)
print(f'Registros totales  : {len(df):>12,}')
print(f'Fabricantes        : {df["make"].nunique():>12,}')
print(f'Modelos únicos     : {df["model"].nunique():>12,}')
print(f'Unidades vendidas  : {df["units_sold"].sum():>12,.0f}')
print(f'Revenue total      : ${df["revenue"].sum()/1e9:>10.1f}B')
print(f'Precio promedio    : ${df["avg_price"].mean():>10,.0f}')
print('=' * 55)
print()

# Ventas por segmento
seg = df.groupby('segment').agg(
    units   = ('units_sold', 'sum'),
    revenue = ('revenue',    'sum'),
    avg_px  = ('avg_price',  'mean'),
).sort_values('revenue', ascending=False)
seg['revenue_B'] = (seg['revenue'] / 1e9).round(1)
seg['share_%']   = (seg['units'] / seg['units'].sum() * 100).round(1)
print('Ventas por Segmento:')
print(seg[['units', 'revenue_B', 'avg_px', 'share_%']].to_string())

         📊 RESUMEN DEL DATASET
Registros totales  :        7,200
Fabricantes        :           15
Modelos únicos     :          120
Unidades vendidas  :   66,945,601
Revenue total      : $    2746.6B
Precio promedio    : $    46,128

Ventas por Segmento:
                units  revenue_B        avg_px  share_%
segment                                                
Mass Market  52761280     1875.6  33941.142438     78.8
SUV/Truck     6752623      342.8  50988.018281     10.1
Luxury        4461413      326.7  72172.990778      6.7
EV/Tech       2970285      201.4  67956.065292      4.4


## 📈 5. Visualizaciones Interactivas

In [8]:
# ────────────────────────────────────────────────────────────────────────────
#  CHART 1: Market Share — Treemap de Revenue por Fabricante
# ────────────────────────────────────────────────────────────────────────────
make_rev = (
    df.groupby(['segment', 'make'])
      .agg(revenue=('revenue', 'sum'), units=('units_sold', 'sum'))
      .reset_index()
)
make_rev['revenue_B']  = (make_rev['revenue'] / 1e9).round(2)
make_rev['share']      = (make_rev['units'] / make_rev['units'].sum() * 100).round(1)

fig1 = px.treemap(
    make_rev,
    path        = [px.Constant('Mercado Total'), 'segment', 'make'],
    values      = 'revenue',
    color       = 'revenue_B',
    color_continuous_scale = 'Blues',
    custom_data = ['revenue_B', 'share'],
    title       = '🗺️ Market Share por Revenue — Todos los Fabricantes (2020-2024)',
)
fig1.update_traces(
    hovertemplate = '<b>%{label}</b><br>Revenue: $%{customdata[0]}B<br>Market share: %{customdata[1]}%<extra></extra>',
    textinfo      = 'label+percent entry',
)
fig1.update_layout(**LAYOUT_DEFAULTS, height=520,
    coloraxis_colorbar=dict(title='Revenue (B USD)'))
fig1.show()
print('💡 Interacción: Haz click en un segmento para hacer zoom')

💡 Interacción: Haz click en un segmento para hacer zoom


In [9]:
# ────────────────────────────────────────────────────────────────────────────
#  CHART 2: Evolución de Ventas Anuales por Fabricante (Área apilada)
# ────────────────────────────────────────────────────────────────────────────
annual_make = (
    df.groupby(['year', 'make'])
      .agg(units=('units_sold', 'sum'))
      .reset_index()
)

# Ordenar fabricantes por ventas totales
order = annual_make.groupby('make')['units'].sum().sort_values(ascending=False).index.tolist()

fig2 = px.area(
    annual_make,
    x           = 'year',
    y           = 'units',
    color       = 'make',
    category_orders = {'make': order},
    labels      = {'units': 'Unidades Vendidas', 'year': 'Año', 'make': 'Fabricante'},
    title       = '📈 Evolución de Ventas por Fabricante (2020-2024)',
    color_discrete_sequence = px.colors.qualitative.Bold,
)
fig2.update_layout(**LAYOUT_DEFAULTS, height=500,
    hovermode  = 'x unified',
    legend     = dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
)
fig2.update_xaxes(tickmode='linear', tick0=2020, dtick=1)
fig2.show()
print('💡 Interacción: Haz click en la leyenda para mostrar/ocultar fabricantes')

💡 Interacción: Haz click en la leyenda para mostrar/ocultar fabricantes


In [10]:
# ────────────────────────────────────────────────────────────────────────────
#  CHART 3: EV vs ICE — Crecimiento Comparativo
# ────────────────────────────────────────────────────────────────────────────
ev_trend = (
    df.groupby(['year', 'drive_type'])
      .agg(units=('units_sold', 'sum'), revenue=('revenue', 'sum'))
      .reset_index()
)

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Unidades Vendidas por Tipo', 'Revenue por Tipo (USD)'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}]]
)

colors = {'Electric': '#00D4FF', 'Hybrid': '#FFE66D', 'Gasoline': '#FF6B6B', 'Diesel': '#A8A8A8'}

for dtype in ev_trend['drive_type'].unique():
    sub = ev_trend[ev_trend['drive_type'] == dtype]
    c   = colors.get(dtype, '#ffffff')
    fig3.add_trace(go.Bar(x=sub['year'], y=sub['units'],
                          name=dtype, marker_color=c, legendgroup=dtype), row=1, col=1)
    fig3.add_trace(go.Bar(x=sub['year'], y=sub['revenue']/1e9,
                          name=dtype, marker_color=c, legendgroup=dtype, showlegend=False), row=1, col=2)

fig3.update_layout(**LAYOUT_DEFAULTS, height=480,
    title_text  = '⚡ Gasoline vs Hybrid vs Electric — Tendencia 2020-2024',
    barmode     = 'stack',
    legend      = dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig3.update_yaxes(title_text='Unidades', row=1, col=1)
fig3.update_yaxes(title_text='Revenue (B USD)', row=1, col=2)
fig3.show()
print('💡 Insight: Los vehículos eléctricos crecen a un ritmo 4x mayor que el mercado total')

💡 Insight: Los vehículos eléctricos crecen a un ritmo 4x mayor que el mercado total


In [11]:
# ────────────────────────────────────────────────────────────────────────────
#  CHART 4: Heatmap de Estacionalidad — Ventas Mensuales
# ────────────────────────────────────────────────────────────────────────────
MONTH_NAMES = {1:'Ene', 2:'Feb', 3:'Mar', 4:'Abr', 5:'May', 6:'Jun',
               7:'Jul', 8:'Ago', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dic'}

seasonal = (
    df.groupby(['year', 'month'])
      .agg(units=('units_sold', 'sum'))
      .reset_index()
)
seasonal['month_name'] = seasonal['month'].map(MONTH_NAMES)
heat_pivot = seasonal.pivot(index='year', columns='month_name', values='units')
# Ordenar meses correctamente
heat_pivot = heat_pivot[[MONTH_NAMES[m] for m in range(1, 13)]]

fig4 = go.Figure(go.Heatmap(
    z             = heat_pivot.values,
    x             = heat_pivot.columns.tolist(),
    y             = heat_pivot.index.tolist(),
    colorscale    = 'RdYlGn',
    hovertemplate = 'Año: %{y}<br>Mes: %{x}<br>Unidades: %{z:,.0f}<extra></extra>',
    text          = heat_pivot.values,
    texttemplate  = '%{text:,.0f}',
))
fig4.update_layout(**LAYOUT_DEFAULTS, height=360,
    title = '📅 Estacionalidad de Ventas — Heatmap Mensual por Año',
    xaxis_title = 'Mes',
    yaxis_title = 'Año',
)
fig4.show()
print('💡 Insight: Noviembre-Diciembre y Julio-Agosto son los meses pico')

💡 Insight: Noviembre-Diciembre y Julio-Agosto son los meses pico


In [12]:
# ────────────────────────────────────────────────────────────────────────────
#  CHART 5: Scatter — Precio vs Volumen (Bubble Chart)
# ────────────────────────────────────────────────────────────────────────────
make_summary = (
    df.groupby(['make', 'segment'])
      .agg(
          avg_price = ('avg_price',  'mean'),
          units     = ('units_sold', 'sum'),
          revenue   = ('revenue',    'sum'),
      )
      .reset_index()
)
make_summary['revenue_B'] = (make_summary['revenue'] / 1e9).round(1)

fig5 = px.scatter(
    make_summary,
    x           = 'avg_price',
    y           = 'units',
    size        = 'revenue_B',
    color       = 'segment',
    text        = 'make',
    labels      = {'avg_price': 'Precio Promedio (USD)', 'units': 'Unidades Totales (2020-2024)',
                   'revenue_B': 'Revenue (B USD)', 'segment': 'Segmento'},
    title       = '💰 Precio vs Volumen vs Revenue — Posicionamiento de Fabricantes',
    color_discrete_sequence = [PRIMARY, SECONDARY, ACCENT, '#A8FF78'],
    size_max    = 60,
)
fig5.update_traces(textposition='top center', marker=dict(opacity=0.85, line=dict(width=1, color='white')))
fig5.update_layout(**LAYOUT_DEFAULTS, height=540,
    xaxis = dict(tickformat='$,', title='Precio Promedio (USD)'),
    yaxis = dict(title='Unidades Totales Vendidas'),
)
fig5.show()
print('💡 Insight: Toyota y Ford maximizan volumen. Tesla lidera revenue por unidad en el segmento EV')

💡 Insight: Toyota y Ford maximizan volumen. Tesla lidera revenue por unidad en el segmento EV


In [13]:
# ────────────────────────────────────────────────────────────────────────────
#  CHART 6: Línea de tiempo — Top 5 Fabricantes (ventas mensuales)
# ────────────────────────────────────────────────────────────────────────────
top5 = make_summary.nlargest(5, 'units')['make'].tolist()
monthly_top5 = (
    df[df['make'].isin(top5)]
      .groupby(['date', 'make'])
      .agg(units=('units_sold', 'sum'))
      .reset_index()
)

fig6 = px.line(
    monthly_top5,
    x      = 'date',
    y      = 'units',
    color  = 'make',
    labels = {'units': 'Unidades/Mes', 'date': 'Fecha', 'make': 'Fabricante'},
    title  = '📊 Ventas Mensuales — Top 5 Fabricantes',
    color_discrete_sequence = [PRIMARY, SECONDARY, ACCENT, '#A8FF78', '#FF9F43'],
    markers = True,
)
fig6.update_layout(**LAYOUT_DEFAULTS, height=460,
    hovermode = 'x unified',
    legend    = dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
)
fig6.update_traces(line=dict(width=2.5), marker=dict(size=5))

# Anotar el dip de COVID
fig6.add_annotation(
    x='2020-04-01', y=monthly_top5[monthly_top5['make']=='Toyota']['units'].min(),
    text='⚠️ Impacto COVID-19', showarrow=True, arrowhead=2,
    font=dict(color=SECONDARY, size=11),
    arrowcolor=SECONDARY, ax=60, ay=-40
)
fig6.show()
print('💡 Interacción: Arrastra para hacer zoom, doble click para reset')

💡 Interacción: Arrastra para hacer zoom, doble click para reset


In [14]:
# ────────────────────────────────────────────────────────────────────────────
#  CHART 7: Dashboard KPI — Sunburst por Segmento > Fabricante > Drive Type
# ────────────────────────────────────────────────────────────────────────────
sunburst_df = (
    df.groupby(['segment', 'make', 'drive_type'])
      .agg(units=('units_sold', 'sum'))
      .reset_index()
)

fig7 = px.sunburst(
    sunburst_df,
    path   = ['segment', 'make', 'drive_type'],
    values = 'units',
    title  = '🌐 Composición del Mercado — Segmento › Fabricante › Propulsión',
    color_discrete_sequence = px.colors.qualitative.Set3,
)
fig7.update_traces(
    hovertemplate = '<b>%{label}</b><br>Unidades: %{value:,.0f}<br>%{percentRoot:.1%} del total<extra></extra>',
    textinfo = 'label+percent entry',
)
fig7.update_layout(**LAYOUT_DEFAULTS, height=580)
fig7.show()
print('💡 Interacción: Haz click en cualquier arco para hacer drill-down')

💡 Interacción: Haz click en cualquier arco para hacer drill-down


## 🧠 6. Insights & Conclusiones

In [15]:
# ── KPIs Finales ──────────────────────────────────────────────────────────────
total_units   = df['units_sold'].sum()
total_rev     = df['revenue'].sum()
ev_units      = df[df['drive_type']=='Electric']['units_sold'].sum()
ev_share_2024 = (df[(df['drive_type']=='Electric') & (df['year']==2024)]['units_sold'].sum() /
                 df[df['year']==2024]['units_sold'].sum() * 100)
yoy_growth    = ((df[df['year']==2024]['units_sold'].sum() /
                  df[df['year']==2020]['units_sold'].sum()) - 1) * 100
top_maker     = df.groupby('make')['units_sold'].sum().idxmax()
top_seg_rev   = df.groupby('segment')['revenue'].sum().idxmax()

print('=' * 60)
print('              🏆 KEY FINDINGS — RESUMEN EJECUTIVO')
print('=' * 60)
print(f'  📦 Unidades totales analizadas : {total_units:>12,.0f}')
print(f'  💰 Revenue total (5 años)      : ${total_rev/1e9:>10.1f}B USD')
print(f'  🥇 Fabricante líder en volumen : {top_maker:>15}')
print(f'  💎 Segmento mayor revenue      : {top_seg_rev:>15}')
print(f'  ⚡ Share EV en 2024            : {ev_share_2024:>13.1f}%')
print(f'  📈 Crecimiento 2020-2024       : {yoy_growth:>12.1f}%')
print('=' * 60)
print()
print('📋 RECOMENDACIONES ESTRATÉGICAS:')
print('  1. 🔋 Priorizar portfolio EV — crecimiento 4x vs mercado')
print('  2. 🗓️  Concentrar campañas en Nov-Dic y Jul-Ago (picos estacionales)')
print('  3. 💡 Segmento Luxury: menor volumen pero mayor margen unitario')
print('  4. 🌎 Toyota y Ford dominan pero tienen baja penetración EV — oportunidad')
print('  5. ⚡ Tesla crece con precio premium — validar elasticidad de demanda')

              🏆 KEY FINDINGS — RESUMEN EJECUTIVO
  📦 Unidades totales analizadas :   66,945,601
  💰 Revenue total (5 años)      : $    2746.6B USD
  🥇 Fabricante líder en volumen :          Toyota
  💎 Segmento mayor revenue      :     Mass Market
  ⚡ Share EV en 2024            :          18.9%
  📈 Crecimiento 2020-2024       :         27.7%

📋 RECOMENDACIONES ESTRATÉGICAS:
  1. 🔋 Priorizar portfolio EV — crecimiento 4x vs mercado
  2. 🗓️  Concentrar campañas en Nov-Dic y Jul-Ago (picos estacionales)
  3. 💡 Segmento Luxury: menor volumen pero mayor margen unitario
  4. 🌎 Toyota y Ford dominan pero tienen baja penetración EV — oportunidad
  5. ⚡ Tesla crece con precio premium — validar elasticidad de demanda


## 💾 7. Exportar Resultados

In [16]:
from google.colab import files

# Guardar dataset procesado
df.to_csv('vehicle_sales_dataset.csv', index=False)
make_summary.to_csv('make_summary.csv', index=False)

# Exportar gráficos como HTML interactivos
figs = [fig1, fig2, fig3, fig4, fig5, fig6, fig7]
names = ['treemap', 'area_annual', 'ev_vs_ice', 'heatmap_seasonal',
         'scatter_price_volume', 'monthly_top5', 'sunburst']

for fig, name in zip(figs, names):
    fig.write_html(f'chart_{name}.html')

print('✅ Archivos exportados:')
print('  📄 vehicle_sales_dataset.csv')
print('  📄 make_summary.csv')
for name in names:
    print(f'  🌐 chart_{name}.html')

# Descargar CSV principal
files.download('vehicle_sales_dataset.csv')
print('\n✅ Dataset descargado.')

✅ Archivos exportados:
  📄 vehicle_sales_dataset.csv
  📄 make_summary.csv
  🌐 chart_treemap.html
  🌐 chart_area_annual.html
  🌐 chart_ev_vs_ice.html
  🌐 chart_heatmap_seasonal.html
  🌐 chart_scatter_price_volume.html
  🌐 chart_monthly_top5.html
  🌐 chart_sunburst.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Dataset descargado.


---

## 📎 Stack Técnico
| Herramienta | Versión | Uso |
|---|---|---|
| Python | 3.10+ | Lenguaje base |
| Pandas | 2.x | Manipulación de datos |
| NumPy | 1.24+ | Generación de datos sintéticos |
| Plotly | 5.x | Visualizaciones interactivas |
| Requests | 2.x | Consumo API NHTSA |
| NHTSA vPIC API | — | Catálogo de vehículos (free) |

---
*Proyecto de portfolio — Datos de ventas sintéticos con distribuciones basadas en tendencias reales del mercado automotriz.*  
*API NHTSA: https://vpic.nhtsa.dot.gov/api/ — Gratis, sin autenticación.*